# 📖 Notebook 4: Cache Patterns & Stampede Protection

Notebooks 1–3 answered *where* data lives and *how* it stays in sync across nodes.
This notebook answers the other half of the story:

> When the cache *misses*, how does the data get there — and what happens when
> **everyone misses at the same time**?

We'll cover the four classic **cache access patterns** (cache-aside, read-through,
write-through, write-behind), then tackle the most famous distributed-cache failure
mode: the **cache stampede** (a.k.a. thundering herd).

## Learning Objectives

By the end of this notebook, you'll understand:
- The four cache access patterns and when to use each
- Why naive cache-aside can serve stale data forever
- What a cache stampede is and how it takes down databases
- How to stop a stampede with a **Redis distributed lock** (SET NX)
- Why "negative caching" (caching the absence of a value) matters


## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/distributed-cache
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In this notebook we only need **one** Redis node as our cache — the goal is to
understand *patterns*, not partitioning. We'll use `node-1` as the cache, and
simulate a slow "database" with `time.sleep`.


In [ ]:
import redis
import time
import threading
import uuid
import queue

# Use one Redis node as the cache
cache = redis.Redis(host="localhost", port=6381, decode_responses=True)
cache.flushall()
cache.ping()
print("✅ cache is up (node-1 on port 6381)")


## 🗄️ The Fake Database

To see cache patterns in action, we need a "database" that's slow enough to
notice. This `FakeDB` sleeps 100ms on every read — just like a real SQL query
against a remote Postgres. It also **counts** every read so we can *prove* how
many times the DB was hit.


In [ ]:
class FakeDB:
    """A fake database with latency and a built-in hit counter."""

    def __init__(self, latency_ms: int = 100):
        self.data = {}
        self.latency_ms = latency_ms
        self.read_count = 0
        self.write_count = 0
        self._lock = threading.Lock()

    def read(self, key: str):
        time.sleep(self.latency_ms / 1000)
        with self._lock:
            self.read_count += 1
        return self.data.get(key)

    def write(self, key: str, value: str):
        time.sleep(self.latency_ms / 1000)
        with self._lock:
            self.write_count += 1
        self.data[key] = value

    def reset_counters(self):
        self.read_count = 0
        self.write_count = 0


db = FakeDB(latency_ms=100)

# Seed some initial data
db.data["product:42"] = "iPhone 15 — $999"
db.data["product:99"] = "AirPods Pro — $249"
db.reset_counters()

print(f"💾 DB seeded with {len(db.data)} rows, latency={db.latency_ms}ms")


---

## Part 1: Cache-Aside (Lazy Loading) — The Default Pattern

In **cache-aside**, the *application* owns the logic:

1. Check the cache.
2. On **hit** → return the value.
3. On **miss** → read from the DB, write it to the cache, return it.

This is by far the most common pattern. Redis + Python apps almost always use it.

```
  app ──► cache.get(key)
           │
      miss ▼
       db.read(key) ──► cache.set(key, value) ──► return value
```


### ❌ Bad: Cache-Aside Without a TTL or Invalidation

The naive version "just works" — until the DB changes. Then the cache serves
stale data **forever**.


In [ ]:
def cache_aside_bad_get(key: str) -> str:
    """Cache-aside with NO ttl and NO invalidation on writes. Will serve stale data."""
    cached = cache.get(key)
    if cached is not None:
        return cached
    value = db.read(key)
    if value is not None:
        cache.set(key, value)  # No TTL — lives forever
    return value


def cache_aside_bad_update(key: str, new_value: str):
    """Update the DB but forget to touch the cache."""
    db.write(key, new_value)
    # BUG: cache still has the old value


cache.flushall()
db.reset_counters()

# First read — MISS, hits DB
v1 = cache_aside_bad_get("product:42")
print(f"1. First read  → {v1}  (db reads so far: {db.read_count})")

# Update the underlying data
cache_aside_bad_update("product:42", "iPhone 15 — $899 (SALE)")
print(f"2. DB updated to: {db.data['product:42']}")

# Read again — cache returns the STALE old value
v2 = cache_aside_bad_get("product:42")
print(f"3. Second read → {v2}  ❌ STALE — cache never got the memo")

# Guard rail: this cell is supposed to REPRODUCE the bug. If the second read
# ever comes back fresh, the "bad" version silently works and Part 1 teaches
# nothing — so fail loudly instead.
assert v2 == v1, f"expected the stale old value {v1!r}, got {v2!r}"
assert v2 != db.data["product:42"], "cache-aside-without-invalidation did not go stale"
assert cache.ttl("product:42") == -1, "the bad version is supposed to set NO ttl"
print("   (verified: cached value != DB value, and the key has no TTL to save us)")

### ✅ Good: Cache-Aside with TTL + Invalidate-on-Write

Two small fixes eliminate 99% of staleness problems:

1. **TTL**: every cache entry expires after N seconds → worst-case staleness is bounded.
2. **Invalidate-on-write**: when the DB changes, delete the cache key so the next
   read repopulates it fresh.

The TTL acts as a *safety net* even if invalidation is missed (e.g. a bug, a
crashed process, or a write from outside your app).


In [ ]:
CACHE_TTL = 60  # seconds


def cache_aside_good_get(key: str) -> str:
    cached = cache.get(key)
    if cached is not None:
        return cached
    value = db.read(key)
    if value is not None:
        cache.set(key, value, ex=CACHE_TTL)  # TTL safety net
    return value


def cache_aside_good_update(key: str, new_value: str):
    db.write(key, new_value)
    cache.delete(key)  # Invalidate → next read will repopulate fresh


cache.flushall()
db.reset_counters()

# Warm the cache
cache_aside_good_get("product:42")

# Update the DB and invalidate
cache_aside_good_update("product:42", "iPhone 15 — $899 (SALE)")

# Next read — MISS (because we invalidated), then fresh value
v = cache_aside_good_get("product:42")
print(f"After invalidate-on-write → {v}  ✅ fresh")
print(f"Total DB reads: {db.read_count} (two: initial warm + repopulate after invalidate)")

# Guard rail: fresh value, exactly two DB reads, and the TTL safety net is armed.
assert v == db.data["product:42"], f"cache served {v!r} but the DB holds {db.data['product:42']!r}"
assert db.read_count == 2, f"expected 2 DB reads (warm + repopulate), got {db.read_count}"
assert 0 < cache.ttl("product:42") <= CACHE_TTL, "the repopulated entry should carry a TTL"

---

## Part 2: Read-Through — Let the Cache Own the Lookup

**Cache-aside** puts the "on miss → read DB" logic in the *application*.
**Read-through** moves that logic *into the cache itself* (or a library wrapper).

```
  app ──► cache.get(key)   # that's it
             │
      miss ▼ (transparent to app)
           db.read(key) ──► populate cache ──► return
```

The app only ever calls `cache.get(...)`. Cleaner API, but the wrapper has to
know how to query the DB. Redis itself doesn't do this natively, but client
libraries (e.g. Spring Cache, Django's `cache.get_or_set`) implement it on top.


In [ ]:
class ReadThroughCache:
    """Wraps a cache + DB so callers only see cache.get()."""

    def __init__(self, cache_client, db, ttl: int = 60):
        self.cache = cache_client
        self.db = db
        self.ttl = ttl

    def get(self, key: str):
        cached = self.cache.get(key)
        if cached is not None:
            return cached
        # Miss — the wrapper (not the app) calls the DB
        value = self.db.read(key)
        if value is not None:
            self.cache.set(key, value, ex=self.ttl)
        return value


rt = ReadThroughCache(cache, db, ttl=60)

cache.flushall()
db.reset_counters()

# App only ever touches the wrapper
first = rt.get("product:42")   # miss → DB
second = rt.get("product:42")  # hit  → cache
print(first)
print(second)
print(f"DB reads: {db.read_count} (should be 1)")

# Guard rail: the whole value of read-through is that the second call never
# reaches the DB. Assert it rather than eyeballing the printed number.
assert db.read_count == 1, f"second get() should have been a cache hit, got {db.read_count} DB reads"
assert first == second == db.data["product:42"]

---

## Part 3: Write-Through — Keep Cache & DB in Lockstep on Writes

In **write-through**, every write goes to **both** the cache and the DB, synchronously,
before returning to the client.

✅ Cache is always fresh — no invalidation needed.
❌ Every write pays for two hops (cache + DB latency).
❌ Failure semantics depend on the *order* of those two hops.

### ⚠️ Order matters

The naive drawing puts the cache first:

```
  app ──► write(key, v) ──► cache.set(key, v) ──► db.write(key, v) ──► return
                                 └── if THIS fails, the cache now holds a row
                                     that does not exist in the source of truth
```

That failure is unrecoverable by retry: readers will happily serve the phantom
value until its TTL runs out. So write the **source of truth first**:

```
  app ──► write(key, v) ──► db.write(key, v) ──► cache.set(key, v) ──► return
                                                      └── if THIS fails, the next
                                                          read is just a miss and
                                                          repopulates. Much less bad.
```

The code below uses the second ordering — DB first, then cache.

In [ ]:
def write_through(key: str, value: str):
    """DB first, then cache. On DB failure, cache is untouched (safer)."""
    db.write(key, value)          # source of truth first
    cache.set(key, value, ex=60)  # then the cache


cache.flushall()
db.reset_counters()

write_through("product:100", "Mug — $15")

# Read should be a cache HIT immediately
print(f"cache: {cache.get('product:100')}")
print(f"db:    {db.data['product:100']}")
print(f"db reads since write-through: {db.read_count} (should be 0 — no read needed)")

# Guard rail: write-through's promise is "cache and DB agree the instant the
# write returns, with zero read-back". All three parts of that are checkable.
assert cache.get("product:100") == db.data["product:100"] == "Mug — $15"
assert db.read_count == 0, f"write-through should not read the DB, got {db.read_count}"

# And demonstrate the ordering guarantee: if the DB write blows up, the cache
# must NOT be left holding a phantom row.
original_write = db.write
def exploding_write(key, value):
    raise RuntimeError("DB is down")
db.write = exploding_write
try:
    write_through("product:101", "Phantom — $0")
except RuntimeError:
    pass
finally:
    db.write = original_write

assert cache.get("product:101") is None, (
    "DB write failed but the cache was populated anyway — that's the phantom-row bug"
)
assert "product:101" not in db.data
print("✅ DB-first ordering verified: a failed DB write leaves no phantom cache entry.")

---

## Part 4: Write-Behind (Write-Back) — Speed at the Cost of Durability

**Write-behind** writes to the cache *immediately* and queues the DB write to
happen **asynchronously** in the background.

```
  app ──► cache.set(key, v) ──► return (fast!)
                              └── queue ──► db.write(key, v)   (later)
```

✅ Writes feel instant to the app.
❌ If the cache / app crashes before the queue drains → **writes are lost**.
❌ Readers going directly to the DB see stale data during the lag window.

This is how some NoSQL systems and the OS page cache work. It's powerful but
requires durable queues (e.g. Kafka, Redis Streams) to be safe in production.


In [ ]:
write_queue: "queue.Queue[tuple[str, str]]" = queue.Queue()
stop_flag = threading.Event()


def writer_worker():
    """Background worker that drains the queue into the DB."""
    while not stop_flag.is_set() or not write_queue.empty():
        try:
            key, value = write_queue.get(timeout=0.1)
        except queue.Empty:
            continue
        db.write(key, value)  # 100ms latency — but off the hot path
        write_queue.task_done()


def write_behind(key: str, value: str):
    cache.set(key, value, ex=60)
    write_queue.put((key, value))


# Start the background writer
worker = threading.Thread(target=writer_worker, daemon=True)
worker.start()

cache.flushall()
db.reset_counters()

# Fire off 5 writes and measure how fast the app sees them "complete"
NUM_WRITES = 5
sync_equivalent_ms = NUM_WRITES * db.latency_ms
start = time.time()
for i in range(NUM_WRITES):
    write_behind(f"order:{i}", f"order data {i}")
app_elapsed = (time.time() - start) * 1000

print(f"App-visible write time for {NUM_WRITES} writes: {app_elapsed:.1f} ms "
      f"(compare to {NUM_WRITES}×{db.latency_ms}ms = {sync_equivalent_ms}ms sync)")

# Immediately after, the DB hasn't caught up yet
writes_now = db.write_count
depth_now = write_queue.qsize()
print(f"DB writes right now:        {writes_now} (likely 0 or very few)")
print(f"Queue depth right now:      {depth_now}")

# This is the consistency window write-behind buys its speed with: the cache
# already says the order exists, and anyone reading the DB directly does not
# see it yet.
cached_now = cache.get("order:4")
assert cached_now is not None, "write-behind must populate the cache synchronously"
assert "order:4" not in db.data, (
    "the DB should still be behind here — if it isn't, this isn't write-behind"
)
print(f"   ⚠️  cache says order:4 = {cached_now!r}, but the DB has no row for it yet.")

# Guard rail: the whole point is that the app doesn't pay the DB latency.
assert app_elapsed < sync_equivalent_ms / 5, (
    f"write-behind took {app_elapsed:.1f}ms; a synchronous version would be "
    f"~{sync_equivalent_ms}ms, so this is not off the hot path"
)

# Wait for the queue to drain
write_queue.join()
print(f"DB writes after drain:      {db.write_count} (should be {NUM_WRITES})")
assert db.write_count == NUM_WRITES, f"queue drained but only {db.write_count} rows landed"
assert all(f"order:{i}" in db.data for i in range(NUM_WRITES))

stop_flag.set()
worker.join(timeout=2)

> 💡 **Durability warning.** In our demo the queue is just an in-process
> `queue.Queue`. If the process crashes between `write_behind(...)` returning
> and the worker flushing, those writes are gone. Real systems back the queue
> with a durable log (Redis Streams, Kafka, a WAL on disk).


---

## Part 5: The Cache Stampede (Thundering Herd)

A **cache stampede** happens when a *hot* key expires (or was never populated)
and **many concurrent requests miss at the same time**. Every one of them
thinks *"I'll just read the DB and repopulate"*. The DB gets hammered by N
identical queries instead of 1.

This is arguably the #1 reason cache layers take down databases in production.

### Why is it "thundering"?

Imagine 500 web servers, each handling 200 requests/second, all serving a page
that reads `product:iphone`. That key's TTL expires. Suddenly **100,000 requests
per second** all miss and slam the DB with the same query.

### Let's prove it — deterministically

To make the demo reproducible, we'll use a `threading.Barrier` so all workers
line up and start **at exactly the same instant**.


In [ ]:
HOT_KEY = "product:iphone"
NUM_WORKERS = 20

# Make sure the hot key actually exists in the DB so a successful fetch
# will populate the cache — otherwise None is returned and every worker misses.
db.data[HOT_KEY] = "iPhone 15 Pro Max — $1199"


def naive_get(key: str) -> str:
    """Plain cache-aside — no stampede protection."""
    cached = cache.get(key)
    if cached is not None:
        return cached
    # MISS — fetch from DB and populate
    value = db.read(key)
    cache.set(key, value, ex=60)
    return value


def run_stampede(fetcher, label: str) -> tuple[int, list]:
    """Fire NUM_WORKERS simultaneous misses. Returns (db_reads, results)."""
    cache.flushall()
    db.reset_counters()
    barrier = threading.Barrier(NUM_WORKERS)
    results = []

    def worker():
        barrier.wait()  # synchronize all workers
        results.append(fetcher(HOT_KEY))

    threads = [threading.Thread(target=worker) for _ in range(NUM_WORKERS)]
    start = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    elapsed = (time.time() - start) * 1000

    db_reads = db.read_count
    print(f"[{label}]")
    print(f"  workers: {NUM_WORKERS}")
    print(f"  DB reads triggered: {db_reads}  {'🔥 STAMPEDE!' if db_reads > 1 else '✅'}")
    print(f"  DB work consumed:   {db_reads * db.latency_ms} ms of query time")
    print(f"  total elapsed: {elapsed:.0f} ms")
    print()
    return db_reads, results


naive_reads, naive_results = run_stampede(naive_get, "naive cache-aside")

# Guard rail: the failure must actually REPRODUCE. If a lucky interleaving let
# some workers hit the cache, the "fix" below would be measured against nothing.
# (>= NUM_WORKERS - 1 rather than == so one unlucky thread that gets scheduled
# a full 100ms late, after the leader already populated the cache, doesn't turn
# a correct lab into a flaky one. Anything below that means the barrier broke.)
assert naive_reads >= NUM_WORKERS - 1, (
    f"expected ~all {NUM_WORKERS} workers to stampede the DB, but only "
    f"{naive_reads} reads happened — the barrier is not lining them up"
)
assert len(naive_results) == NUM_WORKERS
assert all(r == db.data[HOT_KEY] for r in naive_results)

You should see all 20 workers hit the DB — that's **20× the load** the DB
should have seen. In production with 10,000 workers, that's 10,000× the load
on a single query, frequently enough to cause an outage.

### 🛡️ Fix: Single-Flight with a Redis Distributed Lock

The idea: when a miss happens, only **one** worker is allowed to talk to the
DB. The others wait briefly and retry the cache.

We implement this with Redis's atomic `SET key value NX EX ttl`:

- `NX` → set **only if** the key doesn't exist (atomic "acquire lock").
- `EX ttl` → the lock **auto-expires** if the holder crashes (no dead locks).
- We store a random **token** as the value so only the holder can release it.
- Release uses a tiny **Lua script** for atomic compare-and-delete — otherwise
  a slow worker could delete a later worker's lock.


In [ ]:
# Lua script: delete the lock only if the token matches.
# Runs atomically inside Redis — no TOCTOU race.
RELEASE_LOCK_LUA = """
if redis.call('GET', KEYS[1]) == ARGV[1] then
    return redis.call('DEL', KEYS[1])
else
    return 0
end
"""
release_lock = cache.register_script(RELEASE_LOCK_LUA)


def single_flight_get(key: str, lock_ttl_ms: int = 5000, wait_ms: int = 50) -> str:
    """Cache-aside with a distributed lock so only one worker recomputes a miss."""
    # 1. Fast path — cache hit
    cached = cache.get(key)
    if cached is not None:
        return cached

    # 2. Miss — try to become the "leader" who recomputes
    lock_key = f"lock:{key}"
    token = str(uuid.uuid4())

    for _ in range(100):  # bounded retry
        acquired = cache.set(lock_key, token, nx=True, px=lock_ttl_ms)
        if acquired:
            try:
                # 3. Double-check: maybe another worker populated while we waited
                cached = cache.get(key)
                if cached is not None:
                    return cached
                # 4. We're the leader — read DB and populate cache
                value = db.read(key)
                if value is not None:
                    cache.set(key, value, ex=60)
                return value
            finally:
                release_lock(keys=[lock_key], args=[token])
        else:
            # 5. Someone else is recomputing — wait briefly and retry the cache
            time.sleep(wait_ms / 1000)
            cached = cache.get(key)
            if cached is not None:
                return cached
            # cache still empty → loop and try to acquire ourselves
    raise RuntimeError("single_flight_get: gave up after too many retries")


sf_reads, sf_results = run_stampede(single_flight_get, "single-flight with Redis lock")

# Guard rail: EXACTLY one DB read, and every worker still got the right answer.
# "Exactly one" is the claim — not "fewer", not "usually one".
assert sf_reads == 1, f"single-flight should collapse {NUM_WORKERS} misses into 1 DB read, got {sf_reads}"
assert len(sf_results) == NUM_WORKERS
assert all(r == db.data[HOT_KEY] for r in sf_results), (
    "a waiting worker returned something other than the recomputed value"
)
print(f"✅ {NUM_WORKERS} concurrent misses → {sf_reads} DB read "
      f"({naive_reads}× less DB work than the naive version).")

With the lock, **exactly 1 DB read** happens no matter how many workers pile
in — the other 19 wait ~50ms and then read the freshly populated cache.
That's the whole trick.

> ⚠️ **The hole in this implementation.** Look at step 4 again: the leader only
> calls `cache.set(...)` `if value is not None`. For a key that genuinely doesn't
> exist in the DB, the leader populates nothing, releases the lock, and the next
> waiter becomes leader and queries the DB too — the stampede comes right back,
> just serialised. Part 6 (negative caching) is what closes this hole; in a real
> client you want both, with the sentinel written under the same lock.

### 💭 Other stampede mitigations (briefly)

- **Probabilistic early recomputation (XFetch):** refresh the key *before*
  it expires, with a probability that grows as TTL approaches 0. Avoids the
  moment-of-expiry cliff. See the XFetch paper for the math.
- **Stale-while-revalidate:** serve the *stale* value immediately and refresh
  in the background. Popular in HTTP caches and CDNs.
- **Request coalescing in-process:** Python's `asyncio` + a `dict` of in-flight
  futures is a lightweight version of single-flight for a single process.
- **TTL jitter:** the lock above protects *one* key. It does nothing when ten
  thousand *different* keys expire in the same second — measured next.

For most Redis-backed services, the distributed lock above is the standard
answer.

---

## Part 5b: TTL Jitter — the Stampede the Lock Can't Stop

The single-flight lock solves the *hot key* stampede: many requests, one key.
There is a second, sneakier version: **many keys, one instant**.

Warm a cold cache — after a deploy, a failover, a `FLUSHALL` — and ten thousand
keys get populated inside a few seconds, **all with the same TTL**. Exactly one
TTL later they all expire together, and the miss burst repeats. Worse, it is
*self-perpetuating*: each burst re-populates the whole working set at once, so
the next generation is just as sharp. A fixed TTL turns a one-off cold start
into a permanent, synchronised drumbeat on your database.

Every key here has a different name, so every one takes a *different* lock.
Single-flight cannot help.

The fix is one line: **randomise each TTL by ±10%** so the expiries smear out.

```python
cache.set(key, value, ex=int(ttl * random.uniform(0.9, 1.1)))
```

Let's measure how much that one line is worth.

In [ ]:
import random
from collections import Counter

NUM_KEYS = 10_000
BASE_TTL = 300         # 5 minutes
WARM_WINDOW_S = 5      # a cold start repopulates the working set in ~5 seconds
GENERATIONS = 3        # follow the burst through three expiry cycles
DB_CAPACITY = 500      # misses/second this database can absorb before it browns out


def simulate_expiry_bursts(jitter_frac: float, seed: int = 11) -> list[int]:
    """
    Return the peak misses-per-second for each expiry generation.

    We don't need to wait 5 minutes to see this: a key that is written at time t
    with TTL x expires at t + x, so tracking expiry timestamps is enough.
    """
    rng = random.Random(seed)
    # Cold start: the whole working set is warmed inside WARM_WINDOW_S.
    next_expiry = [rng.uniform(0, WARM_WINDOW_S) for _ in range(NUM_KEYS)]

    peaks = []
    for _ in range(GENERATIONS):
        for i in range(NUM_KEYS):
            ttl = BASE_TTL * (1 + rng.uniform(-jitter_frac, jitter_frac))
            next_expiry[i] += ttl
        # Bucket expiries into 1-second bins; the fullest bin is the peak miss rate.
        peaks.append(max(Counter(int(t) for t in next_expiry).values()))
    return peaks


flat = simulate_expiry_bursts(jitter_frac=0.0)
jittered = simulate_expiry_bursts(jitter_frac=0.10)

print(f"📊 Peak cache misses per second ({NUM_KEYS:,} keys, TTL={BASE_TTL}s)")
print("=" * 66)
print(f"{'Generation':<14} {'fixed TTL':>14} {'TTL ± 10%':>14}   vs DB capacity")
print("-" * 66)
for g, (f, j) in enumerate(zip(flat, jittered), start=1):
    verdict = "🔴 overloaded" if f > DB_CAPACITY else "🟢"
    verdict_j = "🔴 overloaded" if j > DB_CAPACITY else "🟢 fine"
    print(f"  expiry #{g:<5} {f:>12,}/s {j:>12,}/s   fixed {verdict} · jittered {verdict_j}")

print()
print(f"   DB can absorb ~{DB_CAPACITY}/s.")
print(f"   Fixed TTL peaks at {flat[0]:,}/s — {flat[0]/DB_CAPACITY:.1f}× over budget, forever.")
print(f"   ±10% jitter peaks at {jittered[0]:,}/s and keeps flattening: "
      f"{jittered[-1]:,}/s by generation {GENERATIONS}.")

# Guard rails.
# 1. The unjittered burst must reproduce, and must NOT decay on its own --
#    that self-perpetuating property is the whole reason jitter is needed.
assert flat[0] > DB_CAPACITY * 2, f"fixed-TTL burst should overwhelm the DB, got {flat[0]}/s"
assert len(set(flat)) == 1, f"a fixed TTL should reproduce the SAME burst forever, got {flat}"
# 2. Jitter must cut the first peak by a large factor...
assert jittered[0] < flat[0] / 5, (
    f"±10% jitter should cut the peak several-fold: {jittered[0]}/s vs {flat[0]}/s"
)
# 3. ...and must keep smearing it out generation over generation.
assert jittered[-1] < jittered[0], f"jittered peaks should keep decaying, got {jittered}"
assert jittered[0] < DB_CAPACITY, "jittered peak should fit inside the DB's budget"

print()
print("🔑 One `random.uniform(0.9, 1.1)` turned a permanent "
      f"{flat[0]/DB_CAPACITY:.0f}× overload into a non-event.")
print("   Jitter is the cheapest stampede defence there is — and unlike the lock,")
print("   it works on the many-keys-one-instant case. Use both.")

---

## Part 6: Negative Caching — Cache the *Absence* of a Value

What happens when someone asks for `user:does-not-exist` a million times? Every
single call hits the DB, gets `None`, and returns. The cache never helps
because we only store *hits*.

**Negative caching** fixes this by storing a **sentinel value** that means
"this key really doesn't exist" — with a **shorter TTL** than normal entries,
so newly-created records show up quickly.

> ⚠️ Don't cache a bare Python `None` — you can't distinguish it from "not in
> cache at all". Use an explicit marker like `"__MISS__"`.


In [ ]:
MISS_SENTINEL = "__MISS__"
POSITIVE_TTL = 300     # 5 minutes for real values
NEGATIVE_TTL = 30      # 30 seconds for "not found" — short so new records appear quickly


def get_with_negative_cache(key: str):
    cached = cache.get(key)
    if cached == MISS_SENTINEL:
        return None           # we've recorded that this key doesn't exist
    if cached is not None:
        return cached         # real cached value
    # Miss — go to DB
    value = db.read(key)
    if value is None:
        cache.set(key, MISS_SENTINEL, ex=NEGATIVE_TTL)
    else:
        cache.set(key, value, ex=POSITIVE_TTL)
    return value


cache.flushall()
db.reset_counters()

# Ask for a key that doesn't exist — 5 times
ghosts = [get_with_negative_cache("user:ghost") for _ in range(5)]

print(f"DB reads for 5 lookups of a missing key: {db.read_count}  (should be 1)")

# Guard rails: one DB read, callers still see None (not the sentinel string),
# and the sentinel expires sooner than a real value would.
assert db.read_count == 1, f"negative caching should collapse 5 misses into 1 DB read, got {db.read_count}"
assert ghosts == [None] * 5, f"the sentinel must never leak to callers, got {ghosts}"
assert cache.get("user:ghost") == MISS_SENTINEL
assert 0 < cache.ttl("user:ghost") <= NEGATIVE_TTL < POSITIVE_TTL

# And a real value must still work through the same path.
real = get_with_negative_cache("product:42")
assert real == db.data["product:42"]
assert cache.ttl("product:42") > NEGATIVE_TTL, "real values get the long TTL"
print("✅ sentinel returns None to callers, expires fast, and real values keep the long TTL.")

---

## 🌍 Real-World Examples

| System | Pattern | Notes |
|--------|---------|-------|
| **Facebook TAO / Memcached** | Cache-aside + lease tokens | Leases are Facebook's flavor of single-flight — prevents stampedes for hot celebrity profiles. |
| **Instagram feed** | Read-through + stale-while-revalidate | Feed service serves slightly stale data and refreshes asynchronously. |
| **Amazon DynamoDB DAX** | Write-through | DAX writes to cache and DynamoDB synchronously for transparency. |
| **Linux page cache** | Write-behind | Dirty pages flush to disk asynchronously — fast, but `sync` exists for a reason. |
| **Twitter timelines** | Negative caching | Profile lookups for deleted / non-existent accounts are cached to protect the user service. |

## 🧪 Try It Yourself

1. **Increase the worker count** to 200 in the stampede demo. The naive version
   will do ~200 DB reads; single-flight still does 1. See the DB latency drop.
2. **Kill the lock holder** mid-recompute (e.g. add `raise Exception()` inside
   the leader branch). Verify the lock still auto-expires so the system recovers.
3. **Break write-through** by failing the DB write (`raise` inside `db.write`
   temporarily). What ends up in the cache? How would you detect this?
4. **Close the single-flight hole**: make the leader cache `MISS_SENTINEL` when
   `db.read` returns `None`, then re-run the stampede against a key that does
   *not* exist in the DB. How many DB reads before and after?
5. **Tune the jitter**: try ±1%, ±10%, ±30% in Part 5b. How much jitter do you
   need before the peak fits under `DB_CAPACITY` on the very first expiry?

## 📝 Key Takeaways

| Pattern | Who owns DB logic | Fresh on write? | Risk |
|---------|------------------|-----------------|------|
| Cache-aside | App | Only with TTL + invalidate | Stale without invalidation |
| Read-through | Cache wrapper | Same as cache-aside | Same, just tidier API |
| Write-through | App → DB → cache (sync) | Always | Slower writes; cache-first ordering leaves phantom rows |
| Write-behind | App → cache (sync) → DB (async) | Eventually | **Data loss on crash**; DB readers see a stale window |

| Problem | Fix |
|---------|-----|
| Stale cache | TTL + invalidate-on-write |
| Hot-key stampede (many requests, one key) | Redis lock (SET NX EX) with token + Lua release |
| Mass-expiry stampede (many keys, one instant) | TTL jitter — randomise each TTL by ±10% |
| Misses hammering DB | Negative cache with sentinel + short TTL |

## ➡️ Next

You've now seen the full distributed-cache stack: partitioning (NB1), coherence (NB2),
consistent hashing (NB3), and access patterns + stampede protection (NB4). Together
these are what Redis Cluster, Memcached, DynamoDB DAX, and Facebook TAO all do
under the hood.

In [ ]:
# Cleanup
cache.flushall()
print("🧹 Cache flushed.")
